# BT4012: Student-selected autoencoder + XGBoost comparison

**11 October 2026.** The student selected this fixed semi-supervised representation experiment after reviewing how the unlabelled transactions would contribute. Keep the previously selected four chronological checks and original XGBoost settings. No automatic search, blending, model selection or Kaggle upload is performed.

The goal is to test whether unknown transactions help us learn useful transaction summaries. These are genuinely new fitted models. A result must be measured before it can be called an improvement over the existing classifier or its recorded public score of 0.94265.

## 1. How unlabelled transactions enter learning

A **denoising autoencoder** learns to reconstruct a clean feature vector from a slightly noisy version. This task needs the transaction's features but no licit/illicit answer. Its middle layer supplies 32 learned numbers describing the transaction. A mean squared reconstruction error supplies one more number. XGBoost receives these 33 summaries plus the original 165 features, giving 198 prediction features, and learns only from confirmed labels.

This is unsupervised representation pretraining followed by supervised classification. The combined pipeline uses labelled and unlabelled examples; the autoencoder itself does not learn fraud labels. An unknown transaction is never assigned a guessed answer. A low reconstruction error indicates a familiar feature pattern, not confirmed legitimacy. A high error is not confirmed fraud.

The assumption under test is that patterns learned from the additional transactions also help order licit and illicit transactions in future periods. The majority of unlabelled rows may instead teach patterns irrelevant to fraud or specific to time. Preserving the original features gives the classifier access to its previous clues, but does not guarantee an improvement.

General method references only: [semi-supervised learning](https://scikit-learn.org/stable/modules/semi_supervised.html), [denoising representation learning](https://www.jmlr.org/papers/v11/vincent10a.html). No external competition data, labels or solution is used.

## 2. Student-selected settings and controls

Compare **original XGBoost**, **known-only autoencoder + XGBoost**, and **all-row autoencoder + XGBoost**. The known-only encoder tests the changed representation without using unknown rows. The all-row encoder uses both known and unknown feature rows in the permitted training period. Each scaler follows the same row restriction as its encoder. This comparison assesses the entire addition of unlabelled data, including scaling and pretraining; it does not isolate one mechanism.

Encoder: 165 → 128 → 32. Decoder: 32 → 128 → 165. Hidden layers use ReLU; latent and reconstruction outputs are linear. Standardize using pretraining rows only and clip to ±10. Add Gaussian noise with standard deviation 0.1 during training. Use 30 complete passes, batch size 512, Adam learning rate 0.001, weight decay 0.0001, seed 42 and two CPU threads. Both conditions use the same number of passes; the larger all-row dataset therefore entails more gradient updates, a comparison limitation.

Keep original XGBoost settings: 15 leaves, 127 bins, learning rate 0.05, lambda 10, minimum child weight 3, full row/feature sampling, positive-class weight 1, seed 42, at most 400 trees, and 60-round stopping. No number is changed after seeing outer scores.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

data_locations = [Path.cwd() / 'Data Set', Path.cwd().parent / 'Data Set',
                  Path('/Users/mlvlo/BT4012/Kaggle/Data Set')]
DATA_DIR = next((p for p in data_locations if (p / 'Train Data.csv').is_file()), None)
assert DATA_DIR is not None, 'Set DATA_DIR to the supplied competition files.'
OUTPUT_DIR = Path.cwd() / 'semisupervised_autoencoder_results'
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'encoders').mkdir(exist_ok=True)
FEATURES = [f'feat_{i}' for i in range(1, 166)]
AE_FEATURES = [f'ae_{i:02d}' for i in range(1, 33)] + ['ae_reconstruction_mse']
CHECKS = [(14, 15, 28), (17, 18, 31), (19, 20, 33), (21, 22, 35)]
FAMILIES = ['original_xgboost', 'ae_known_only', 'ae_all_rows']
SEED, CPU_THREADS, EPOCHS, BATCH_SIZE = 42, 2, 30, 512
NOISE_SD, ADAM_RATE, WEIGHT_DECAY = 0.1, 0.001, 0.0001
MAXIMUM_ROUNDS, STOPPING_PATIENCE = 400, 60
assert 'torch' not in sys.modules, 'Use a fresh process for this notebook.'

train_path = DATA_DIR / 'Train Data.csv'
train = pd.read_csv(train_path, usecols=['txId', 'time_step', 'label'] + FEATURES)
assert train.txId.is_unique and train.time_step.between(1, 35).all()
assert np.isfinite(train[FEATURES].to_numpy()).all()
known = train.loc[train.label.notna()].copy()
assert known.label.isin([0, 1]).all()
known['label'] = known.label.astype(int)
assert known.label.nunique() == 2
encoder_records = []
worker_call_number = 0
torch_version = None

def progress(**details):
    temporary = OUTPUT_DIR / 'progress.tmp'
    temporary.write_text(json.dumps(details))
    temporary.replace(OUTPUT_DIR / 'progress.json')

print('Training rows:', len(train), '; known:', len(known), '; unknown:', int(train.label.isna().sum()))
print('XGBoost:', xgboost.__version__, '; CPU threads:', CPU_THREADS)


Training rows: 141772 ; known: 31235 ; unknown: 110537
XGBoost: 3.2.0 ; CPU threads: 2


## 3. Train feature reconstruction without fraud answers

The denoising target is the clean standardized feature vector. The noise is added only during pretraining. We transform transactions without noise during XGBoost fitting and prediction. Save the encoder, scaler and loss curve for each fitting stage. The loss curve documents reconstruction learning; it does not choose a fraud model or a number of passes.

The autoencoder training function accepts feature rows and does not read their fraud labels. Label availability chooses rows only for the known-only comparison. Neither condition receives validation answers.

**Runtime correction:** loading the neural and tree libraries together caused a native OpenMP threading crash on this Mac before any validation result was produced. The notebook therefore writes its own encoder worker from the source below and runs it in a fresh Python process. PyTorch and XGBoost do not share a process. No model setting changes or additional package installation are needed. The worker is generated by this notebook, so it is included in reproducibility rather than being an external dependency.

In [2]:
WORKER_SOURCE = r'''
from pathlib import Path
import json
import sys
import time
import numpy as np
import torch
from torch import nn

configuration = json.loads(sys.argv[1])
torch.set_num_threads(configuration['threads'])
torch.set_num_interop_threads(configuration['threads'])
torch.use_deterministic_algorithms(True)

class TransactionAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(165, 128), nn.ReLU(), nn.Linear(128, 32))
        self.decoder = nn.Sequential(nn.Linear(32, 128), nn.ReLU(), nn.Linear(128, 165))

    def forward(self, noisy_features):
        return self.decoder(self.encoder(noisy_features))

def encode(clean, model):
    batches = []
    model.eval()
    with torch.no_grad():
        for first in range(0, len(clean), configuration['batch_size']):
            inputs = torch.from_numpy(clean[first:first + configuration['batch_size']])
            latent = model.encoder(inputs)
            error = (inputs - model.decoder(latent)).square().mean(dim=1, keepdim=True)
            batches.append(torch.cat([latent, error], dim=1).numpy())
    return np.concatenate(batches, axis=0)

clean_array = np.load(configuration['input_path'], allow_pickle=False)
assert clean_array.dtype == np.float32 and np.isfinite(clean_array).all()
started = time.perf_counter()
if configuration['action'] == 'fit':
    torch.manual_seed(configuration['seed'])
    model = TransactionAutoencoder()
    optimizer = torch.optim.Adam(model.parameters(), lr=configuration['learning_rate'],
                                 weight_decay=configuration['weight_decay'])
    shuffle_generator = torch.Generator().manual_seed(configuration['seed'])
    clean = torch.from_numpy(clean_array)
    curve = []
    for epoch in range(1, configuration['epochs'] + 1):
        model.train()
        order = torch.randperm(len(clean), generator=shuffle_generator)
        total_loss = 0.0
        for first in range(0, len(clean), configuration['batch_size']):
            batch = clean[order[first:first + configuration['batch_size']]]
            noisy = batch + configuration['noise_sd'] * torch.randn_like(batch)
            optimizer.zero_grad()
            loss = (model(noisy) - batch).square().mean()
            assert torch.isfinite(loss), 'Non-finite reconstruction loss.'
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * len(batch)
        curve.append(total_loss / len(clean))
        progress_path = Path(configuration['progress_path'])
        temporary = progress_path.with_suffix('.tmp')
        temporary.write_text(json.dumps(dict(
            phase='encoder pretraining', family=configuration['family'],
            cutoff=configuration['end'], stage=configuration['stage'],
            epoch=epoch, epochs=configuration['epochs'], rows=len(clean),
            seconds=round(time.perf_counter() - started, 2))))
        temporary.replace(progress_path)
    assert all(torch.isfinite(parameter).all() for parameter in model.parameters())
    torch.save(model.state_dict(), configuration['weights_path'])
    restored = TransactionAutoencoder()
    restored.load_state_dict(torch.load(configuration['weights_path'], map_location='cpu', weights_only=True))
    probe = clean_array[:min(len(clean_array), 1024)]
    assert np.array_equal(encode(probe, model), encode(probe, restored))
    print(json.dumps(dict(torch_version=torch.__version__, curve=curve,
                          saved_encoder_reload_verified=True)))
else:
    assert configuration['action'] == 'encode'
    model = TransactionAutoencoder()
    model.load_state_dict(torch.load(configuration['weights_path'], map_location='cpu', weights_only=True))
    summaries = encode(clean_array, model)
    assert summaries.shape == (len(clean_array), 33) and np.isfinite(summaries).all()
    np.save(configuration['output_path'], summaries, allow_pickle=False)
    print(json.dumps(dict(torch_version=torch.__version__)))
'''
WORKER_PATH = OUTPUT_DIR / 'encoder_worker.py'
WORKER_PATH.write_text(WORKER_SOURCE)


def run_encoder_worker(clean_array, configuration):
    global worker_call_number, torch_version
    worker_call_number += 1
    input_path = OUTPUT_DIR / f'.encoder_input_{worker_call_number}.npy'
    output_path = OUTPUT_DIR / f'.encoder_output_{worker_call_number}.npy'
    np.save(input_path, clean_array, allow_pickle=False)
    configuration = dict(configuration, input_path=str(input_path), output_path=str(output_path),
                         threads=CPU_THREADS, batch_size=BATCH_SIZE)
    result = subprocess.run([sys.executable, str(WORKER_PATH), json.dumps(configuration)],
                            capture_output=True, text=True, timeout=600)
    assert result.returncode == 0, f'Encoder worker failed: {result.stderr}'
    details = json.loads(result.stdout)
    torch_version = details['torch_version']
    summaries = np.load(output_path, allow_pickle=False) if output_path.exists() else None
    input_path.unlink()
    if output_path.exists():
        output_path.unlink()
    return details, summaries


def pretraining_rows(family, end):
    assert family in ['ae_known_only', 'ae_all_rows']
    rows = train.loc[train.time_step <= end]
    if family == 'ae_known_only':
        rows = rows.loc[rows.label.notna()]
    assert len(rows) and rows.time_step.max() <= end
    return rows


def fit_encoder(feature_rows, *, family, end, stage):
    raw = feature_rows[FEATURES].to_numpy(dtype=np.float32)
    assert raw.shape[1] == 165 and np.isfinite(raw).all()
    scaler = StandardScaler().fit(raw)
    clean_array = np.clip(scaler.transform(raw), -10, 10).astype(np.float32)
    started = time.perf_counter()
    prefix = f'{family}_{end}_{stage}'
    weights_path = OUTPUT_DIR / 'encoders' / f'{prefix}.pt'
    details, _ = run_encoder_worker(clean_array, dict(
        action='fit', weights_path=str(weights_path), epochs=EPOCHS, seed=SEED,
        noise_sd=NOISE_SD, learning_rate=ADAM_RATE, weight_decay=WEIGHT_DECAY,
        family=family, end=end, stage=stage, progress_path=str(OUTPUT_DIR / 'progress.json')))
    assert details['saved_encoder_reload_verified']
    curve = details['curve']
    np.savez_compressed(OUTPUT_DIR / 'encoders' / f'{prefix}_scaler.npz',
                        mean=scaler.mean_, scale=scaler.scale_, var=scaler.var_,
                        n_samples_seen=np.asarray(scaler.n_samples_seen_))
    pd.DataFrame({'epoch': np.arange(1, EPOCHS + 1), 'noisy_reconstruction_mse': curve}).to_csv(
        OUTPUT_DIR / 'encoders' / f'{prefix}_loss.csv', index=False)
    encoder_records.append(dict(family=family, train_end=end, stage=stage,
                                rows=len(feature_rows), known_rows=int(feature_rows.label.notna().sum()),
                                unknown_rows=int(feature_rows.label.isna().sum()),
                                first_epoch_mse=curve[0], last_epoch_mse=curve[-1],
                                seconds=round(time.perf_counter() - started, 3)))
    pd.DataFrame(encoder_records).to_csv(OUTPUT_DIR / 'encoder_training.csv', index=False)
    return weights_path, scaler


def augmented_features(rows, encoder, scaler):
    clean = np.clip(scaler.transform(rows[FEATURES].to_numpy(dtype=np.float32)), -10, 10).astype(np.float32)
    _, summaries = run_encoder_worker(clean, dict(action='encode', weights_path=str(encoder)))
    assert summaries.shape == (len(rows), 33) and np.isfinite(summaries).all()
    extra = pd.DataFrame(summaries, index=rows.index, columns=AE_FEATURES)
    features = pd.concat([rows[FEATURES], extra], axis=1)
    assert features.shape == (len(rows), 198) and features.index.equals(rows.index)
    assert not any(name in features.columns for name in ['label', 'time_step', 'txId', 'index'])
    return features


## 4. Chronological stopping, followed by refitting

For a cutoff of 14, the initial scaler and encoder see rows only through step 11. Known labels in steps 1–11 fit XGBoost; known labels in 12–14 select its tree count. Then refit the scaler and encoder on permitted rows through 14, and refit XGBoost on all known labels through 14 for the selected tree count. Predict steps 15–28 without fitting on their features or answers. Other cutoffs use the same last-three-period stopping rule.

Refitting the encoder changes its representation. The selected count is therefore a training recipe carried to a fresh representation, not a guarantee of the refitted model's optimal count. This is recorded as a limitation rather than adjusted using outer answers.

Four checks overlap and have already informed development. Their scores cannot establish an independent hidden-test performance estimate. The original control must reproduce the previously recorded probabilities.

In [3]:
def make_xgboost(rounds, *, stopping=False):
    return XGBClassifier(
        objective='binary:logistic', n_estimators=int(rounds), learning_rate=0.05,
        tree_method='hist', device='cpu', grow_policy='lossguide', max_depth=0,
        max_leaves=15, max_bin=127, min_child_weight=3, reg_lambda=10,
        subsample=1.0, colsample_bytree=1.0, scale_pos_weight=1.0,
        eval_metric='auc', early_stopping_rounds=STOPPING_PATIENCE if stopping else None,
        n_jobs=CPU_THREADS, random_state=SEED,
    )


def fit_pipeline(family, end):
    learning = known.loc[known.time_step <= end]
    fitting = learning.loc[learning.time_step <= end - 3]
    stopping = learning.loc[learning.time_step > end - 3]
    assert fitting.time_step.max() < stopping.time_step.min()
    assert fitting.label.nunique() == stopping.label.nunique() == 2
    if family == 'original_xgboost':
        fit_features, stop_features = fitting[FEATURES], stopping[FEATURES]
        encoder, scaler = None, None
    else:
        inner_rows = pretraining_rows(family, end - 3)
        assert set(inner_rows.txId).isdisjoint(stopping.txId)
        encoder, scaler = fit_encoder(inner_rows, family=family, end=end - 3, stage='inner')
        fit_features = augmented_features(fitting, encoder, scaler)
        stop_features = augmented_features(stopping, encoder, scaler)
    progress(phase='XGBoost stopping', family=family, cutoff=end)
    with threadpool_limits(limits=CPU_THREADS):
        initial = make_xgboost(MAXIMUM_ROUNDS, stopping=True)
        initial.fit(fit_features, fitting.label,
                    eval_set=[(stop_features, stopping.label)], verbose=False)
        rounds = int(initial.best_iteration + 1)
        assert 1 <= rounds <= MAXIMUM_ROUNDS
        assert abs(roc_auc_score(stopping.label, initial.predict_proba(stop_features)[:, 1])
                   - float(initial.best_score)) < 1e-12
    curve = initial.evals_result()['validation_0']['auc']
    pd.DataFrame({'round': np.arange(1, len(curve) + 1), 'inner_roc_auc': curve}).to_csv(
        OUTPUT_DIR / f'stopping_curve_{family}_{end}.csv', index=False)
    if family == 'original_xgboost':
        refit_features = learning[FEATURES]
        pretraining_known, pretraining_unknown = 0, 0
    else:
        full_rows = pretraining_rows(family, end)
        encoder, scaler = fit_encoder(full_rows, family=family, end=end, stage='refit')
        refit_features = augmented_features(learning, encoder, scaler)
        pretraining_known = int(full_rows.label.notna().sum())
        pretraining_unknown = int(full_rows.label.isna().sum())
    progress(phase='XGBoost refitting', family=family, cutoff=end, rounds=rounds)
    with threadpool_limits(limits=CPU_THREADS):
        refitted = make_xgboost(rounds)
        refitted.fit(refit_features, learning.label)
    return dict(family=family, end=end, model=refitted, encoder=encoder, scaler=scaler,
                rounds=rounds, learning_rows=len(learning),
                pretraining_known=pretraining_known, pretraining_unknown=pretraining_unknown)


def predict_pipeline(pipeline, rows):
    features = (rows[FEATURES] if pipeline['family'] == 'original_xgboost'
                else augmented_features(rows, pipeline['encoder'], pipeline['scaler']))
    with threadpool_limits(limits=CPU_THREADS):
        return pipeline['model'].predict_proba(features)[:, 1]


## 5. Run the student-selected comparison

Each model predicts exactly the same checking transactions. The saved arrays retain identifiers, labels, periods and probabilities for independent recalculation. Positive differences mean an improvement over the original classifier. No best seed, weight or feature count is selected using these results.

In [4]:
records, saved = [], {}
started_all = time.perf_counter()
for end, first, last in CHECKS:
    checking = known.loc[known.time_step.between(first, last)]
    assert known.loc[known.time_step <= end].time_step.max() < checking.time_step.min()
    assert set(train.loc[train.time_step <= end].txId).isdisjoint(checking.txId)
    saved[f'labels_{end}'] = checking.label.to_numpy()
    saved[f'txIds_{end}'] = checking.txId.to_numpy()
    saved[f'time_steps_{end}'] = checking.time_step.to_numpy()
    for family in FAMILIES:
        started = time.perf_counter()
        pipeline = fit_pipeline(family, end)
        probabilities = predict_pipeline(pipeline, checking)
        assert np.isfinite(probabilities).all() and np.all((probabilities >= 0) & (probabilities <= 1))
        auc = float(roc_auc_score(checking.label, probabilities))
        saved[f'{family}_{end}'] = probabilities
        row = dict(model=family, train_end=end, validation_start=first, validation_end=last,
                   learning_rows=pipeline['learning_rows'], checking_rows=len(checking),
                   pretraining_known=pipeline['pretraining_known'],
                   pretraining_unknown=pipeline['pretraining_unknown'],
                   rounds=pipeline['rounds'], auc=auc,
                   seconds=round(time.perf_counter() - started, 3))
        records.append(row)
        print(f'{family}: learn through {end}; check {first}–{last}; AUC={auc:.6f}; '
              f'trees={pipeline["rounds"]}; unknown pretraining rows={pipeline["pretraining_unknown"]:,}', flush=True)
        pd.DataFrame(records).to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
        np.savez_compressed(OUTPUT_DIR / 'validation_probabilities.npz', **saved)
    progress(phase='validation cutoff complete', cutoff=end)

results = pd.DataFrame(records)
comparison = results.pivot(index='train_end', columns='model', values='auc')[FAMILIES]
comparison['all_rows_minus_original'] = comparison.ae_all_rows - comparison.original_xgboost
comparison['all_rows_minus_known_only'] = comparison.ae_all_rows - comparison.ae_known_only
summary = comparison[FAMILIES].agg(['mean', 'min']).T
comparison.to_csv(OUTPUT_DIR / 'comparison.csv')
summary.to_csv(OUTPUT_DIR / 'summary.csv')
print('\nFour chronological checks:')
print(comparison.round(6).to_string())
print('\nMean and worst full-block AUC:')
print(summary.round(6).to_string())
print('Comparison seconds:', round(time.perf_counter() - started_all, 2))
display(comparison.round(6))
display(summary.round(6))


original_xgboost: learn through 14; check 15–28; AUC=0.968521; trees=185; unknown pretraining rows=0
ae_known_only: learn through 14; check 15–28; AUC=0.969808; trees=177; unknown pretraining rows=0
ae_all_rows: learn through 14; check 15–28; AUC=0.956440; trees=108; unknown pretraining rows=56,094
original_xgboost: learn through 17; check 18–31; AUC=0.925611; trees=336; unknown pretraining rows=0
ae_known_only: learn through 17; check 18–31; AUC=0.908671; trees=337; unknown pretraining rows=0
ae_all_rows: learn through 17; check 18–31; AUC=0.906331; trees=352; unknown pretraining rows=64,134
original_xgboost: learn through 19; check 20–33; AUC=0.901582; trees=398; unknown pretraining rows=0
ae_known_only: learn through 19; check 20–33; AUC=0.887385; trees=276; unknown pretraining rows=0
ae_all_rows: learn through 19; check 20–33; AUC=0.893463; trees=357; unknown pretraining rows=68,482
original_xgboost: learn through 21; check 22–35; AUC=0.915330; trees=381; unknown pretraining rows=0

model,original_xgboost,ae_known_only,ae_all_rows,all_rows_minus_original,all_rows_minus_known_only
train_end,,,,,
14,0.968521,0.969808,0.956440,-0.012082,-0.013369
17,0.925611,0.908671,0.906331,-0.019280,-0.002340
19,0.901582,0.887385,0.893463,-0.008120,0.006077
21,0.915330,0.887894,0.888125,-0.027205,0.000231


,mean,min
model,,
original_xgboost,0.927761,0.901582
ae_known_only,0.913440,0.887385
ae_all_rows,0.911090,0.888125


## 6. Inspect the difficult periods

The fourteen-step pooled AUC is the main comparison. Per-period AUC helps describe where an approach improves or declines. It is missing when the period contains only one label class. The overlapping outer checks must not be pooled by duplicating the same transactions.

In [5]:
step_records = []
for end, first, last in CHECKS:
    labels, steps = saved[f'labels_{end}'], saved[f'time_steps_{end}']
    for step in np.unique(steps):
        selected = steps == step
        row = dict(train_end=end, validation_step=int(step), known_labels=int(selected.sum()),
                   illicit=int(labels[selected].sum()))
        for family in FAMILIES:
            row[family] = (float(roc_auc_score(labels[selected], saved[f'{family}_{end}'][selected]))
                           if len(np.unique(labels[selected])) == 2 else np.nan)
        row['all_rows_minus_original'] = row['ae_all_rows'] - row['original_xgboost']
        step_records.append(row)
per_step = pd.DataFrame(step_records)
per_step.to_csv(OUTPUT_DIR / 'per_time_step.csv', index=False)
display(per_step.loc[per_step.train_end == 21].round(6))


,train_end,validation_step,known_labels,illicit,original_xgboost,ae_known_only,ae_all_rows,all_rows_minus_original
42,21,22,1763,158,0.998833,0.997989,0.998434,-0.000398
43,21,23,1187,53,0.957622,0.952863,0.955542,-0.002080
44,21,24,1126,137,0.795488,0.764623,0.767434,-0.028053
45,21,25,594,118,0.999484,0.999519,0.999501,0.000018
46,21,26,517,96,0.946779,0.930201,0.939776,-0.007002
47,21,27,206,24,0.999542,0.999542,0.999313,-0.000229
48,21,28,284,85,1.000000,1.000000,1.000000,0.000000
49,21,29,1174,329,0.775619,0.727951,0.727393,-0.048226
50,21,30,524,83,0.987323,0.986340,0.989755,0.002431
51,21,31,710,106,0.772820,0.740863,0.726681,-0.046139


## 7. Fit the selected experimental conditions on the full training period

Preparing separate experimental CSVs does not select either condition as the final submission. Both fixed conditions are fitted through step 35. Steps 33–35 first choose the tree count, then all allowed training-period features pretrain a fresh encoder and all 31,235 confirmed labels fit XGBoost. The all-row encoder uses all 141,772 training transactions, including 110,537 unknown transactions. The known-only control excludes unknown rows from both scaling and reconstruction learning.

Test features are transformed only after fitting. No test features or hidden answers fit the encoder, scaler or classifier. Reload every saved component and check that it produces exactly the same predictions. Model files, scaler values, loss curves and the architecture in this notebook allow reproduction.

In [6]:
test = pd.read_csv(DATA_DIR / 'Test Data.csv')
template = pd.read_csv(DATA_DIR / 'Sample Submission' / 'BT4012 Competition Submission.csv')
assert test.txId.is_unique and test['index'].is_unique and template['index'].is_unique
assert len(test) == len(template) and set(test['index']) == set(template['index'])
assert known.time_step.max() < test.time_step.min()
assert set(train.txId).isdisjoint(test.txId)
assert np.isfinite(test[FEATURES].to_numpy()).all()
final_records, test_predictions = [], {}
for family in ['ae_known_only', 'ae_all_rows']:
    pipeline = fit_pipeline(family, 35)
    probabilities = predict_pipeline(pipeline, test)
    assert np.isfinite(probabilities).all() and np.all((probabilities >= 0) & (probabilities <= 1))
    mapping = pd.Series(probabilities, index=test['index'])
    submission = template[['index']].copy()
    submission['target'] = submission['index'].map(mapping)
    assert submission.columns.tolist() == ['index', 'target']
    assert submission['index'].equals(template['index']) and submission.target.notna().all()
    submission_path = OUTPUT_DIR / f'submission_{family}_experimental.csv'
    model_path = OUTPUT_DIR / f'xgboost_{family}_experimental.json'
    submission.to_csv(submission_path, index=False, float_format='%.17g')
    pipeline['model'].save_model(model_path)

    # The independent worker reloads the saved encoder for each prediction and
    # also verified its saved state against its in-memory state after fitting.
    restored_encoder = OUTPUT_DIR / 'encoders' / f'{family}_35_refit.pt'
    with np.load(OUTPUT_DIR / 'encoders' / f'{family}_35_refit_scaler.npz') as stored:
        restored_scaler = StandardScaler()
        restored_scaler.mean_ = stored['mean']
        restored_scaler.scale_ = stored['scale']
        restored_scaler.var_ = stored['var']
        restored_scaler.n_samples_seen_ = stored['n_samples_seen']
        restored_scaler.n_features_in_ = len(FEATURES)
    restored_classifier = XGBClassifier(n_jobs=CPU_THREADS)
    restored_classifier.load_model(model_path)
    restored = dict(pipeline, encoder=restored_encoder, scaler=restored_scaler, model=restored_classifier)
    assert np.array_equal(predict_pipeline(restored, test), probabilities)
    csv = pd.read_csv(submission_path)
    assert csv['index'].equals(template['index'])
    assert np.allclose(csv.target, submission.target, rtol=0, atol=1e-12)
    test_predictions[family] = probabilities
    final_records.append(dict(family=family, rounds=pipeline['rounds'],
                              supervised_rows=pipeline['learning_rows'],
                              encoder_known_rows=pipeline['pretraining_known'],
                              encoder_unknown_rows=pipeline['pretraining_unknown'],
                              test_rows=len(test), submission_path=str(submission_path)))
    print('Full-data candidate:', family, '; trees:', pipeline['rounds'],
          '; unknown encoder rows:', pipeline['pretraining_unknown'],
          '; reloaded model and CSV checks passed.', flush=True)
np.savez_compressed(OUTPUT_DIR / 'test_probabilities.npz',
                    txIds=test.txId.to_numpy(), indices=test['index'].to_numpy(), **test_predictions)
pd.DataFrame(final_records).to_csv(OUTPUT_DIR / 'final_training.csv', index=False)
display(pd.DataFrame(final_records))


Full-data candidate: ae_known_only ; trees: 267 ; unknown encoder rows: 0 ; reloaded model and CSV checks passed.
Full-data candidate: ae_all_rows ; trees: 249 ; unknown encoder rows: 110537 ; reloaded model and CSV checks passed.


,family,rounds,supervised_rows,encoder_known_rows,encoder_unknown_rows,test_rows,submission_path
0,ae_known_only,267,31235,31235,0,15329,/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a90...
1,ae_all_rows,249,31235,31235,110537,15329,/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a90...


## 8. Verify and record assistance

The student chose the fixed autoencoder + XGBoost comparison and retained the four chronological checks. OpenAI Codex implemented and ran those choices, verified the saved components and calculations, and documented the measured result. No final model or experimental submission is automatically selected.

Further material changes need a student decision with an explanation of the hypothesis. Repeatedly trying changes until a reused validation score passes a threshold can overfit the development checks. A public score above 0.94265 or a hidden-test AUC above 0.96 cannot be asserted from these local results.

In [8]:
for end, first, last in CHECKS:
    expected = known.loc[known.time_step.between(first, last)]
    assert np.array_equal(saved[f'txIds_{end}'], expected.txId.to_numpy())
    assert np.array_equal(saved[f'labels_{end}'], expected.label.to_numpy())
    for family in FAMILIES:
        assert abs(roc_auc_score(saved[f'labels_{end}'], saved[f'{family}_{end}'])
                   - comparison.loc[end, family]) < 1e-12
previous_path = Path.cwd() / 'feature_ensemble_results' / 'validation_probabilities.npz'
historical_controls_checked = previous_path.is_file()
if historical_controls_checked:
    with np.load(previous_path) as previous:
        for end, first, last in CHECKS:
            assert np.array_equal(saved[f'labels_{end}'], previous[f'labels_{end}'])
            assert np.array_equal(saved[f'original_xgboost_{end}'], previous[f'original_xgboost_{end}'])
# Historical arrays are an optional project audit, not an input needed to
# reproduce predictions from the supplied competition CSVs and this code.

def file_hash(path):
    hasher = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

encoder_config = dict(encoder_widths=[165, 128, 32], decoder_widths=[32, 128, 165],
                      hidden_activation='ReLU', latent_and_output_activation='linear',
                      preprocessing='StandardScaler fit on respective pretraining rows; clip [-10,10]',
                      epochs=EPOCHS, batch_size=BATCH_SIZE, noise_sd=NOISE_SD,
                      optimizer='Adam', learning_rate=ADAM_RATE, weight_decay=WEIGHT_DECAY,
                      seed=SEED, cpu_threads=CPU_THREADS, features_added=len(AE_FEATURES))
metadata = dict(
    date='2026-10-11', student_decision='train the fixed autoencoder + XGBoost comparison',
    method='unsupervised representation pretraining plus supervised XGBoost',
    input_rows=len(train), known_labels=len(known), unknown_rows=int(train.label.isna().sum()),
    features=FEATURES, representation_features=AE_FEATURES, encoder_config=encoder_config,
    checks=CHECKS, final_training=final_records, summary=summary.to_dict(orient='index'),
    all_rows_mean_change=float(comparison.all_rows_minus_original.mean()),
    all_rows_mean_change_vs_known_only=float(comparison.all_rows_minus_known_only.mean()),
    original_controls_exactly_reproduced=historical_controls_checked, all_saved_components_reload_exactly=True,
    no_outer_features_or_labels_in_fitting=True, no_test_features_in_pretraining=True,
    outer_checks_overlap=True, outer_checks_are_development_evidence=True,
    equal_encoder_epochs_with_unequal_gradient_update_counts=True,
    original_public_auc=0.94265, candidate_public_auc=None,
    student_final_selection='pending review', kaggle_submission_made=False,
    python=platform.python_version(), torch=torch_version, sklearn=sklearn.__version__,
    runtime='isolated PyTorch subprocess; XGBoost in the notebook process',
    initial_openmp_runtime_crash_corrected=True,
    xgboost=xgboost.__version__, pandas=pd.__version__, numpy=np.__version__,
    input_sha256={name:file_hash(DATA_DIR/name) for name in [
        'Train Data.csv', 'Test Data.csv', 'Sample Submission/BT4012 Competition Submission.csv']},
    submission_sha256={family:file_hash(OUTPUT_DIR/f'submission_{family}_experimental.csv')
                       for family in ['ae_known_only','ae_all_rows']},
)
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2))
progress(phase='complete', all_rows_mean_auc=float(summary.loc['ae_all_rows','mean']),
         original_mean_auc=float(summary.loc['original_xgboost','mean']))
print('Chronology, probability alignment, AUC, historical controls, reloaded components and CSV checks passed.')
print('No Kaggle upload or automatic final-model selection was made.')


Chronology, probability alignment, AUC, historical controls, reloaded components and CSV checks passed.
No Kaggle upload or automatic final-model selection was made.


## 9. Measured result and next decision

All code cells executed and passed. The changed optional historical-control audit was re-executed after removing an unnecessary dependency on previous experiment files. Training was not repeated for that documentation/reproducibility fix. The standalone notebook needs only the supplied competition CSVs and installed Python libraries to reproduce predictions. The neural worker source is included in the notebook and regenerated by it.

| Validation steps | Original XGBoost | Known-only encoder | All-row encoder | All-row change |
|---|---:|---:|---:|---:|
| 15–28 | 0.968521 | 0.969808 | 0.956440 | -0.012082 |
| 18–31 | 0.925611 | 0.908671 | 0.906331 | -0.019280 |
| 20–33 | 0.901582 | 0.887385 | 0.893463 | -0.008120 |
| 22–35 | 0.915330 | 0.887894 | 0.888125 | -0.027205 |
| **Mean** | **0.927761** | **0.913440** | **0.911090** | **-0.016672** |

The all-row model improved **0 of four** checks versus the original XGBoost. Its mean change was **-0.016672 AUC**. Against the known-only encoder it improved **2 of four**, with mean change **-0.002350**. These results do not establish a benefit from unknown transactions in this particular representation pipeline. They do not establish that every semi-supervised method would fail.

**AI-assisted recommendation:** Do not spend a remaining submission on this all-row candidate based on the measured results. The original model remains the best recorded public score, **0.94265**. Both new candidates' public/private scores are unknown. No Kaggle upload was made.

The full all-row encoder used **141,772 training transactions**, including **all 110,537 unknown transactions**. Its classifier used **31,235 confirmed labels** and 249 trees. Both full-data experimental CSVs contain **15,329 rows**. Saved encoders, scaler values, XGBoost models, identifier alignment and CSV values were checked. The saved components reproduced identical predictions after reload. Historical original controls exactly reproduced the earlier arrays; AUC was recalculated independently from stored predictions.

The runtime crash occurred before any outer result; it was fixed by separating PyTorch and XGBoost processes, without changing the student-selected model settings. No packages or original input files were modified.

### Interpretation the student should review

1. Unknown transactions provided reconstruction training examples, not invented fraud answers.
2. The known-only comparison distinguishes changing representation from adding unlabelled examples, although the larger dataset also changes gradient-update count and scaling statistics.
3. Learning reconstruction well does not imply learning the fraud distinction. No causal explanation for the lower AUC is established by this comparison alone.
4. Validation features and labels did not fit the scaler, encoder or classifier. Test features were used only after fitting.
5. Four overlapping chronological checks are development evidence. Repeatedly choosing settings from them can overfit these periods.

### Possible next student-selected refinement

A different hypothesis is to give the encoder a **fraud classification objective on confirmed fitting labels**, alongside reconstruction on all fitting rows. A proposed controlled follow-up would keep the same network, 32 summaries, 30 passes and XGBoost settings, add a linear classification head, and train with known-label binary cross-entropy plus **0.1 × reconstruction error**. The head would guide the representation but would not add guessed labels. Unknown rows would contribute only reconstruction. The same chronological exclusions and known-only control would apply.

This follow-up is **proposed, not selected or trained**. Its premise is that a representation guided by confirmed fraud answers may be more relevant than reconstruction alone; this is an inference to test, not a measured improvement. The student must choose the objective and weighting before it runs. No broad automatic search is authorised.

### Assistance record

The student selected the denoising-autoencoder plus original-XGBoost experiment, its fixed proposal and the four chronological checks. OpenAI Codex implemented the comparison, diagnosed the native threading conflict, ran training in isolated processes, verified outputs and drafted this factual entry. The student remains responsible for interpretation, subsequent modelling choices and final submission selection, and must accurately disclose the assistance.
